# V2 atanmis deneyler — Run All yalniz onayli paketi baslatir; taslak paket baslatilmaz.


In [ ]:
import sys, os, json
from pathlib import Path
assert sys.platform == "linux" and "openmc-proje" in sys.executable, "WSL openmc-proje kernelini secin"
PACKAGE = Path.cwd().resolve()
assert PACKAGE.name == "v2_stage_32_ogrenci_04_beavrs_v2_pilot_scope_v1", "Notebook paket dizininden acilmali"
PROJECT = next(p for p in (PACKAGE, *PACKAGE.parents) if (p / "src/reactor_pipeline/v2_students.py").is_file())
assert (PROJECT / "src/reactor_pipeline/v2_students.py").is_file()
sys.path.insert(0, str(PROJECT / "src"))
import openmc
import pandas as pd
from IPython.display import display
from reactor_pipeline import v2_students, v2_student_runner
print("OpenMC", openmc.__version__, "| veri", os.environ.get("OPENMC_CROSS_SECTIONS"))
assert os.environ.get("OPENMC_CROSS_SECTIONS") and Path(os.environ["OPENMC_CROSS_SECTIONS"]).is_file()
info = v2_students.verify_package(PACKAGE, require_approval=False)
manifest = v2_students._read(PACKAGE / "manifest.json")
print("Grup", manifest["group_id"], "| durum", info["state"])
for case in manifest["experiments"]:
    print(case["experiment_id"], case["scenario_id"], case["controls"], case["calculation"])
saved = v2_student_runner.saved_results(PACKAGE)
rows = []
for case in saved["cases"]:
    state = case["state"]
    rows.append({"deney": case["experiment_id"], "senaryo": case["scenario_id"],
                 "durum": state, "tamamlandi": state == "completed",
                 "bekliyor": state == "not_started"})
print("Atanan:", len(rows), "| tamamlanan:", sum(r["tamamlandi"] for r in rows),
      "| bekleyen:", sum(r["bekliyor"] for r in rows),
      "| diger:", sum(not r["tamamlandi"] and not r["bekliyor"] for r in rows))
display(pd.DataFrame(rows))
assignment = json.loads((PROJECT / "configs/v2_student_assignment_v1.json").read_text())
pilot_file = PROJECT / "runs/beavrs_3d/v2_groups" / assignment["pilot_reuse"]["group_id"] / "status.json"
pilot_state = json.loads(pilot_file.read_text())["state"] if pilot_file.is_file() else "bu makinede kayit yok"
print("Danisman pilotu:", len(assignment["pilot_reuse"]["scenario_ids"]),
      "ayri senaryo | grup durumu:", pilot_state, "| yeniden baslatmayin")


In [ ]:
check = v2_student_runner.preflight(PACKAGE)
print("On kontrol:", check)


In [ ]:
current = v2_student_runner.status(PACKAGE)
if current["state"] == "not_started":
    v2_students.verify_package(PACKAGE, require_approval=True)
    current = v2_student_runner.launch(PACKAGE)
print("Durum", current["state"], "PID", current.get("pid"))
print("Log", current.get("group_log"), "durum", current.get("status_file"), "sonuclar", current.get("group_dir"))


## Teslim — hesap başlatmaz
Tamamlanan koşular için WSL terminalinde: `python scripts/v2_transfer.py export student_packages/v2_students_portable_v1/stage_32/ogrenci_04/v2_stage_32_ogrenci_04_beavrs_v2_pilot_scope_v1 deliveries/v2_stage_32_ogrenci_04_beavrs_v2_pilot_scope_v1.zip`. Durum `completed` olmadan teslim tam sayılmaz; varsayılan ZIP statepoint içermez.
